#### **Step 4 - Fusion - Combining Keyword Search + Semantic Search**

- Our goal is to now use our **sparse index (keyword search)** scores and **dense index (semantic search)** to create a fused score that we will use to rank the chunks

- From the two search functions we have a list `[(chunk_dict, score), ...]`

- We cannot simply add the scores since the ranges differ. `BM25` have unbounded ranges, while cosine similiarity will range from -1 to 1, meaning the `BM25` score will always dominate

- Therefore, to combine these 2 metrics into a single score we use `Reciprocal Rank Fusion (RRF)` which ignores the scores completely and **uses only the rank (position) of each chunk in each list**

- `RRF Formula`: **Sum over each ranking method of: 1/(k_constant + rank_in_that_method)**
    - Where **k_constant** is a small constant that dampens the impact of rank differences further down the list

    - Any chunks that do not appear in the list contribute 0 to the score

In [1]:
# assuming we can have multiple retrieval results - we will iterate over the list of retrieval results 
def reciprocal_rank_fusion(retrieval_results, k):

    rrf_scores = {} # store the rrf scores as (chunk_id, score)
    chunks = {} # store the chunks as (chunk_id, chunk_dict) so that we can access chunk dicts using its id for the final return

    # iterate over every score list [(dict, score), ...]
    for result in retrieval_results:
        # iterate into the list - iterate over the list of dicts, access the chunk id and score then compute the RRF contribution
        # using chunk_id accumulate into the dictionary
        for rank, chunk in enumerate(result, start=1):
            rrf_scores[chunk[0]['chunk_id']] = rrf_scores.get(chunk[0]['chunk_id'], 0) + 1 / (k + rank)
            chunks[chunk[0]['chunk_id']] = chunk[0] # add chunk_dict to a dict so we can access by chunk_id

    # we want to return the same (chunk_dict, score) style tuple like previously 
    fusion = [(chunks[key], rrf_scores[key]) for key in rrf_scores]

    # sort the result based on fused score (second element in tuple so x[1])
    return sorted(fusion, key=lambda x: -x[1])

**`Reciprocal Rank Fusion` Intuitive Summary**

- We want to use the **rankings from each retrieval method** to create a combined score using the RRF formula, summing \(1/(k + rank)\) for each chunk across all retrieval methods.

- Accumulate rrf scores in a dict `rrf_scores` as we iterate over every list of scores for every retrieval method - using the rank of each chunk within each retrieval method (the raw score is not used)

- Keep a dictionary of chunks so we can access the chunk_dict using the `chunk_id` - our end goal is to return a list of tuples `(chunk_dict, score)` just like in all the previous retrieval methods 

- Once we have the `rrf_scores` dict, we want to iterate over all keys (`chunk_id`) to obtain the chunk_dict from `chunks` and the score from `rrf_scores` to create a list of tuples `(chunk_dict, score)`(using a list comprehension to create the list) 

**Testing**

In [2]:
from sparse_index import SparseIndex
from dense_index import DenseIndex
from chunking import load_documents, chunk_documents 

docs = load_documents("../data")
chunks = chunk_documents(docs)

sparse = SparseIndex(chunks)
dense = DenseIndex(chunks)

# perform sparse and dense search
sparse_result = sparse.search("How many championships has Faker won?", k=4)
dense_result = dense.search("How many championships has Faker won?", k=4)

# call the rrf function to score chunks using rrf by combining sparse and dense search results 
reciprocal_rank_fusion([sparse_result, dense_result], 60)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[({'chunk_id': 'faker.txt_chunk3',
   'doc_id': 'faker.txt',
   'text': ' player in the history of the game.\n\nOutside of Worlds, Faker has won the Mid-Season Invitational twice, in 2016 and 2017, along with a large number of LCK titles in the Korean domestic league. He is also known for a wide champion pool. Over the years he has been associated with LeBlanc, Ryze, Azir, Syndra, Orianna and many others, and he was among the first mid laners to show how flexible a single role could be.\n\nHis playstyle is defined by mechanical precision, calm decision making and strong map awareness. Teammates and coaches often say his greatest strength is not his hands but his reading of the game: he seems to know where the enemy jungler'},
  0.032266458495966696),
 ({'chunk_id': 'faker.txt_chunk2',
   'doc_id': 'faker.txt',
   'text': ' reinventing his role. He shifted from a pure carry into a strategic leader who set the tempo of the whole team, and he kept adapting to new champions and new patches